In [1]:
import numpy as np

import jax
import jax.numpy as jnp

import seaborn as sns
import matplotlib.pyplot as plt

import pandas as pd

In [2]:
HIDDEN_STATE_SHAPE = 8

In [33]:
def RNN(params, xseq, h=np.zeros(HIDDEN_STATE_SHAPE)):
    Wx, Wh,b, Wout, bout = params
    for x in xseq:
        h = jax.nn.tanh(x*Wx + Wh @ h + b)
    out = jnp.dot(h,Wout) + bout
    return out

In [34]:
params = [np.random.randn(HIDDEN_STATE_SHAPE), np.random.randn(HIDDEN_STATE_SHAPE,HIDDEN_STATE_SHAPE), np.random.randn(HIDDEN_STATE_SHAPE),
         np.random.randn(8), np.random.randn()]

In [35]:
data = pd.read_csv("temperature.csv")
data.Year = pd.to_datetime(data.Year)

# MinMax Normalize
X = data.Anomaly.to_numpy()
M,m = X.max(), X.min()
X = X - m
X = X / (M - m)

In [36]:
Xtrain = [X[i:i+24] for i in range(X.shape[0] - 25)]
Ytrain = [X[i+24] for i in range(X.shape[0] - 25)]

RNN(params, Xtrain[0])

Array(-2.131045, dtype=float32)

In [37]:
for i in range(len(Ytrain)-1):
    assert Ytrain[i] == Xtrain[i+1][-1]

In [52]:
def lossfn(params, Xt, yt):
    yhat = [RNN(params, x).astype(float) for x in Xt]
    err = jnp.array(yhat) - jnp.array(yt)
    return jnp.mean(err**2)

dlossfn = jax.jit(jax.grad(lossfn))

In [ ]:
lossfn(params, Xtrain, Ytrain)